# End-to-End Longformer + XHPDF-style Features (ours)

This notebook is the **trainable / end-to-end** version of your proposed model.

Pipeline:

```text
essay text -> Longformer trainable backbone -> CLS embedding
XHPDF-style discourse features -> StandardScaler
concat [CLS embedding ; discourse features]
MLP/regression head -> score
```

Difference from the previous lightweight notebook:

- Previous: `torch.no_grad()` extracted frozen Longformer embeddings, then trained only the head.
- This version: Longformer is inside the training loop, so gradients update Longformer parameters.

Recommended table name:

```text
End-to-End Longformer + XHPDF-style Features (ours)
```


In [1]:
# ============================================================
# 0. SET PATHS FOR COLAB RUNTIME
# ============================================================
# Upload these 3 CSV files to Colab /content first:
#   - asap_train.csv
#   - asap_val.csv
#   - asap_test.csv
#
# If your filenames are different, edit the paths below.

from pathlib import Path

TRAIN_PATH = Path('/content/asap_train.csv')
VAL_PATH   = Path('/content/asap_val.csv')
TEST_PATH  = Path('/content/asap_test.csv')

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f'Missing file: {p}\n'
            'Please upload the 3 CSV files to Colab /content, or edit TRAIN_PATH/VAL_PATH/TEST_PATH.'
        )

print('Found files:')
print('TRAIN:', TRAIN_PATH)
print('VAL  :', VAL_PATH)
print('TEST :', TEST_PATH)


Found files:
TRAIN: /content/asap_train.csv
VAL  : /content/asap_val.csv
TEST : /content/asap_test.csv


In [2]:
# ============================================================
# 1. INSTALL / IMPORT DEPENDENCIES
# ============================================================
!pip -q install transformers accelerate scikit-learn scipy tqdm

import os
import re
import math
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass


Device: cuda


In [3]:
# ============================================================
# 2. LOAD TRAIN / VAL / TEST CSV FILES
# ============================================================
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print('train_df:', train_df.shape)
print('val_df  :', val_df.shape)
print('test_df :', test_df.shape)
display(train_df.head())

# Edit these if your columns have different names.
ESSAY_SET_COL = 'essay_set'
TEXT_COL = 'essay'
SCORE_COL = 'domain1_score'

required_cols = {ESSAY_SET_COL, TEXT_COL, SCORE_COL}
for name, df in [('train_df', train_df), ('val_df', val_df), ('test_df', test_df)]:
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{name} is missing columns: {missing}. Current columns: {list(df.columns)}')

for df in [train_df, val_df, test_df]:
    df[ESSAY_SET_COL] = df[ESSAY_SET_COL].astype(int)
    df[TEXT_COL] = df[TEXT_COL].fillna('').astype(str)
    df[SCORE_COL] = df[SCORE_COL].astype(float)

print('Essay sets:', sorted(train_df[ESSAY_SET_COL].unique()))


train_df: (9084, 5)
val_df  : (1296, 5)
test_df : (2596, 5)


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75
1,9985,4,The author concluded this sentence because he ...,0.0,0.00
2,3231,2,"I can think of several books that, I would not...",4.0,0.60
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77


Essay sets: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


In [4]:
# ============================================================
# 3. CONFIG: END-TO-END LONGFORMER + XHPDF-STYLE FEATURES
# ============================================================
MODEL_NAME = 'allenai/longformer-base-4096'

# End-to-end training is much heavier than frozen embedding extraction.
# Start conservative. Increase only if GPU allows.
MAX_LENGTH_BY_PROMPT = {
    1: 1024,
    2: 1024,
    3: 768,
    4: 768,
    5: 768,
    6: 768,
    7: 1024,
    8: 1536,
}

# If Colab OOM:
# - set TRAIN_BATCH_SIZE = 1
# - lower MAX_LENGTH_BY_PROMPT, especially prompt 8
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 1   # effective batch = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS

EPOCHS = 10
LR_BACKBONE = 1e-5
LR_HEAD = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 2

USE_AMP = True  # mixed precision on CUDA

OUTPUT_DIR = Path('/content/benchmark_results/ASAP')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_SCORE_RANGES = {
    1: (2, 12),
    2: (1, 6),
    3: (0, 3),
    4: (0, 3),
    5: (0, 4),
    6: (0, 4),
    7: (0, 30),
    8: (0, 60),
}

print('MODEL_NAME:', MODEL_NAME)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Effective train batch:', TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)


MODEL_NAME: allenai/longformer-base-4096
OUTPUT_DIR: /content/benchmark_results/ASAP
Effective train batch: 16


In [5]:
# ============================================================
# 4. QWK HELPERS: ROUND AND VALIDATION-OPTIMIZED THRESHOLDS
# ============================================================
def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true.astype(int), y_pred.astype(int), weights='quadratic')

def clip_and_round(preds, min_score, max_score):
    preds = np.clip(preds, min_score, max_score)
    return np.rint(preds).astype(int)

def apply_thresholds(preds, thresholds, min_score, max_score):
    thresholds = np.asarray(thresholds)
    classes = np.digitize(preds, thresholds) + min_score
    return np.clip(classes, min_score, max_score).astype(int)

def optimize_thresholds(y_true, raw_preds, min_score, max_score):
    n_thresholds = int(max_score - min_score)
    if n_thresholds <= 0:
        return np.array([])

    init_thresholds = np.arange(min_score + 0.5, max_score + 0.5, 1.0)

    def objective(thresholds):
        thresholds = np.sort(thresholds)
        pred_classes = apply_thresholds(raw_preds, thresholds, min_score, max_score)
        return -qwk(y_true, pred_classes)

    result = minimize(
        objective,
        init_thresholds,
        method='Nelder-Mead',
        options={'maxiter': 1000, 'disp': False}
    )
    return np.sort(result.x)

def evaluate_predictions(y_true, raw_preds, min_score, max_score, thresholds=None):
    round_preds = clip_and_round(raw_preds, min_score, max_score)
    out = {'qwk_round': qwk(y_true, round_preds)}
    if thresholds is not None:
        opt_preds = apply_thresholds(raw_preds, thresholds, min_score, max_score)
        out['qwk_opt'] = qwk(y_true, opt_preds)
    return out


In [6]:
# ============================================================
# 5. XHPDF-STYLE DISCOURSE FEATURE EXTRACTION
# ============================================================
CONNECTIVES = {
    'addition': ['also', 'moreover', 'furthermore', 'besides', 'in addition', 'additionally'],
    'contrast': ['however', 'but', 'although', 'though', 'nevertheless', 'on the other hand', 'whereas'],
    'cause': ['because', 'since', 'therefore', 'thus', 'hence', 'consequently', 'as a result'],
    'example': ['for example', 'for instance', 'such as', 'including'],
    'sequence': ['first', 'second', 'third', 'finally', 'then', 'next', 'after', 'before'],
    'conclusion': ['in conclusion', 'to conclude', 'overall', 'in summary', 'to sum up']
}

STOPWORDS = set('''
a an the and or but if while with without of for to in on at by from up down out over under again further then once here there
all any both each few more most other some such no nor not only own same so than too very can will just should now is am are was were be been being
have has had do does did this that these those i you he she it we they me him her us them my your his its our their
'''.split())

def simple_sent_tokenize(text):
    sents = re.split(r'[.!?]+', str(text))
    return [s.strip() for s in sents if s.strip()]

def simple_word_tokenize(text):
    return re.findall(r"\b[a-zA-Z]+(?:'[a-zA-Z]+)?\b", str(text).lower())

def count_phrase(text_lower, phrase):
    return len(re.findall(r'\b' + re.escape(phrase) + r'\b', text_lower))

def safe_div(a, b):
    return float(a) / float(b) if b else 0.0

def flesch_reading_ease_proxy(words, sentences):
    def syllables(word):
        groups = re.findall(r'[aeiouy]+', word.lower())
        return max(1, len(groups))
    n_words = len(words)
    n_sents = max(1, len(sentences))
    n_syll = sum(syllables(w) for w in words)
    return 206.835 - 1.015 * safe_div(n_words, n_sents) - 84.6 * safe_div(n_syll, n_words)

def lexical_chain_proxy_features(words, sentences):
    content_words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    counts = Counter(content_words)
    repeated = {w: c for w, c in counts.items() if c >= 2}

    sent_words = [set(simple_word_tokenize(s)) for s in sentences]
    spans = []
    for w in repeated:
        idxs = [i for i, sw in enumerate(sent_words) if w in sw]
        if idxs:
            spans.append(max(idxs) - min(idxs) + 1)

    large_chains = [w for w, c in counts.items() if c >= 4]
    avg_chain_size = np.mean(list(repeated.values())) if repeated else 0.0
    avg_span = np.mean(spans) if spans else 0.0

    return {
        'lc_avg_chain_size_proxy': avg_chain_size,
        'lc_num_varied_chains_proxy': len(repeated),
        'lc_num_large_chains_proxy': len(large_chains),
        'lc_pct_large_chains_proxy': safe_div(len(large_chains), max(1, len(repeated))),
        'lc_avg_span_proxy': avg_span,
    }

def sentence_overlap_features(sentences):
    sent_sets = []
    for s in sentences:
        words = [w for w in simple_word_tokenize(s) if w not in STOPWORDS and len(w) > 2]
        sent_sets.append(set(words))

    overlaps = []
    jaccards = []
    for a, b in zip(sent_sets[:-1], sent_sets[1:]):
        if not a or not b:
            overlaps.append(0)
            jaccards.append(0.0)
        else:
            inter = len(a & b)
            union = len(a | b)
            overlaps.append(inter)
            jaccards.append(safe_div(inter, union))

    return {
        'adjacent_overlap_mean': np.mean(overlaps) if overlaps else 0.0,
        'adjacent_overlap_max': np.max(overlaps) if overlaps else 0.0,
        'adjacent_jaccard_mean': np.mean(jaccards) if jaccards else 0.0,
    }

def extract_discourse_features_one(text):
    text = str(text)
    text_lower = text.lower()
    sentences = simple_sent_tokenize(text)
    words = simple_word_tokenize(text)

    n_words = len(words)
    n_sentences = len(sentences)
    unique_words = set(words)
    content_words = [w for w in words if w not in STOPWORDS and len(w) > 2]

    feats = {}

    feats['char_count'] = len(text)
    feats['word_count'] = n_words
    feats['sentence_count'] = n_sentences
    feats['avg_sentence_length'] = safe_div(n_words, n_sentences)
    feats['unique_word_count'] = len(unique_words)
    feats['type_token_ratio'] = safe_div(len(unique_words), n_words)
    feats['content_word_ratio'] = safe_div(len(content_words), n_words)

    feats['comma_count'] = text.count(',')
    feats['semicolon_count'] = text.count(';')
    feats['colon_count'] = text.count(':')
    feats['question_count'] = text.count('?')
    feats['exclamation_count'] = text.count('!')
    feats['quote_count'] = text.count('"') + text.count("'")
    feats['punctuation_ratio'] = safe_div(sum(1 for ch in text if ch in ',;:.!?'), max(1, len(text)))

    total_conn = 0
    for group, items in CONNECTIVES.items():
        c = sum(count_phrase(text_lower, item) for item in items)
        feats[f'connective_{group}_count'] = c
        total_conn += c
    feats['connective_total_count'] = total_conn
    feats['connective_per_100_words'] = 100.0 * safe_div(total_conn, n_words)

    feats['flesch_reading_ease_proxy'] = flesch_reading_ease_proxy(words, sentences)

    feats.update(sentence_overlap_features(sentences))
    feats.update(lexical_chain_proxy_features(words, sentences))

    return feats

def extract_discourse_features_df(df, text_col=TEXT_COL):
    rows = [extract_discourse_features_one(x) for x in tqdm(df[text_col].tolist(), desc='Extracting XHPDF-style features')]
    return pd.DataFrame(rows).fillna(0.0)

train_feat_df = extract_discourse_features_df(train_df)
val_feat_df   = extract_discourse_features_df(val_df)
test_feat_df  = extract_discourse_features_df(test_df)

print('Feature shape:', train_feat_df.shape)
display(train_feat_df.head())


Extracting XHPDF-style features:   0%|          | 0/9084 [00:00<?, ?it/s]

Extracting XHPDF-style features:   0%|          | 0/1296 [00:00<?, ?it/s]

Extracting XHPDF-style features:   0%|          | 0/2596 [00:00<?, ?it/s]

Feature shape: (9084, 31)


,char_count,word_count,sentence_count,avg_sentence_length,unique_word_count,type_token_ratio,content_word_ratio,comma_count,semicolon_count,colon_count,...,connective_per_100_words,flesch_reading_ease_proxy,adjacent_overlap_mean,adjacent_overlap_max,adjacent_jaccard_mean,lc_avg_chain_size_proxy,lc_num_varied_chains_proxy,lc_num_large_chains_proxy,lc_pct_large_chains_proxy,lc_avg_span_proxy
0,1194,193,12,16.083333,119,0.616580,0.616580,8,0,0,...,1.036269,41.035805,0.545455,3.0,0.032290,2.600000,20,3,0.150000,6.500000
1,179,32,1,32.000000,28,0.875000,0.500000,1,0,0,...,3.125000,39.523750,0.000000,0.0,0.000000,0.000000,0,0,0.000000,0.000000
2,3915,739,53,13.943396,281,0.380244,0.473613,19,0,0,...,2.976996,80.264050,0.288462,2.0,0.022744,3.093750,64,14,0.218750,23.656250
3,4184,724,50,14.480000,310,0.428177,0.483425,21,0,0,...,2.348066,68.392772,0.122449,1.0,0.011655,2.894737,57,11,0.192982,21.017544
4,1180,255,19,13.421053,116,0.454902,0.439216,0,0,0,...,1.176471,86.052632,0.555556,2.0,0.053108,2.750000,20,4,0.200000,9.250000


In [7]:
# ============================================================
# 6. DATASET AND MODEL CLASSES
# ============================================================
class EssayFeatureDataset(Dataset):
    def __init__(self, df, features, tokenizer, max_length, min_score, max_score):
        self.texts = df[TEXT_COL].tolist()
        self.raw_scores = df[SCORE_COL].values.astype(float)
        self.features = features.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.min_score = min_score
        self.max_score = max_score
        self.score_range = max(1, max_score - min_score)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}

        # Longformer global attention on the first token.
        global_attention_mask = torch.zeros_like(item['attention_mask'])
        global_attention_mask[0] = 1
        item['global_attention_mask'] = global_attention_mask

        item['features'] = torch.tensor(self.features[idx], dtype=torch.float32)
        norm_score = (self.raw_scores[idx] - self.min_score) / self.score_range
        item['labels'] = torch.tensor(norm_score, dtype=torch.float32)
        item['raw_score'] = torch.tensor(self.raw_scores[idx], dtype=torch.float32)
        return item

class LongformerXHPDFRegressor(nn.Module):
    def __init__(self, model_name, feature_dim):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size + feature_dim),
            nn.Dropout(0.1),
            nn.Linear(hidden_size + feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )

    def forward(self, input_ids, attention_mask, global_attention_mask, features):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )
        cls = out.last_hidden_state[:, 0, :]
        x = torch.cat([cls, features], dim=1)
        return self.head(x).squeeze(-1)

def make_optimizer(model):
    no_decay = ['bias', 'LayerNorm.weight', 'layer_norm.weight']
    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        group = {
            'params': [param],
            'weight_decay': 0.0 if any(nd in name for nd in no_decay) else WEIGHT_DECAY,
        }

        if name.startswith('backbone.'):
            group['lr'] = LR_BACKBONE
            backbone_params.append(group)
        else:
            group['lr'] = LR_HEAD
            head_params.append(group)

    return torch.optim.AdamW(backbone_params + head_params)


In [8]:
# ============================================================
# 7. TRAINING / PREDICTION HELPERS
# ============================================================
def run_predict(model, loader, min_score, max_score):
    model.eval()
    preds_norm = []
    gold_raw = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type == 'cuda')):
                pred = model(
                    input_ids=batch['input_ids'],
                    attention_mask=batch['attention_mask'],
                    global_attention_mask=batch['global_attention_mask'],
                    features=batch['features']
                )
            preds_norm.append(pred.detach().cpu().numpy())
            gold_raw.append(batch['raw_score'].detach().cpu().numpy())

    preds_norm = np.concatenate(preds_norm)
    gold_raw = np.concatenate(gold_raw)

    score_range = max(1, max_score - min_score)
    preds_raw = preds_norm * score_range + min_score
    return gold_raw, preds_raw

def train_one_prompt(prompt_id, tr_df, va_df, te_df, tr_feat, va_feat, te_feat):
    min_score, max_score = OFFICIAL_SCORE_RANGES[int(prompt_id)]
    max_length = MAX_LENGTH_BY_PROMPT.get(int(prompt_id), 1024)
    feature_dim = tr_feat.shape[1]

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_ds = EssayFeatureDataset(tr_df, tr_feat, tokenizer, max_length, min_score, max_score)
    val_ds   = EssayFeatureDataset(va_df, va_feat, tokenizer, max_length, min_score, max_score)
    test_ds  = EssayFeatureDataset(te_df, te_feat, tokenizer, max_length, min_score, max_score)

    train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False)

    model = LongformerXHPDFRegressor(MODEL_NAME, feature_dim).to(DEVICE)
    optimizer = make_optimizer(model)

    total_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
    warmup_steps = max(1, int(0.1 * total_steps))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type == 'cuda'))
    loss_fn = nn.MSELoss()

    best_state = None
    best_val_qwk = -999
    patience_counter = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        train_losses = []

        pbar = tqdm(train_loader, desc=f'Prompt {prompt_id} epoch {epoch}')
        for step, batch in enumerate(pbar, start=1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type == 'cuda')):
                pred = model(
                    input_ids=batch['input_ids'],
                    attention_mask=batch['attention_mask'],
                    global_attention_mask=batch['global_attention_mask'],
                    features=batch['features']
                )
                loss = loss_fn(pred, batch['labels'])
                loss = loss / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            train_losses.append(loss.item() * GRAD_ACCUM_STEPS)
            pbar.set_postfix(loss=float(np.mean(train_losses[-20:])))

        val_gold, val_pred_raw = run_predict(model, val_loader, min_score, max_score)
        val_round = clip_and_round(val_pred_raw, min_score, max_score)
        val_qwk_round = qwk(val_gold.astype(int), val_round)

        mean_train_loss = float(np.mean(train_losses))
        history.append({
            'epoch': epoch,
            'train_loss': mean_train_loss,
            'val_qwk_round': val_qwk_round
        })

        print(f'Epoch {epoch} | train_loss={mean_train_loss:.5f} | val_qwk_round={val_qwk_round:.5f}')

        if val_qwk_round > best_val_qwk:
            best_val_qwk = val_qwk_round
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_gold, val_pred_raw = run_predict(model, val_loader, min_score, max_score)
    test_gold, test_pred_raw = run_predict(model, test_loader, min_score, max_score)

    thresholds = optimize_thresholds(val_gold.astype(int), val_pred_raw, min_score, max_score)

    val_eval = evaluate_predictions(val_gold.astype(int), val_pred_raw, min_score, max_score, thresholds)
    test_eval = evaluate_predictions(test_gold.astype(int), test_pred_raw, min_score, max_score, thresholds)

    result = {
        ESSAY_SET_COL: int(prompt_id),
        'method': 'End-to-End Longformer + XHPDF-style Features (ours)',
        'model_name': MODEL_NAME,
        'feature_count': feature_dim,
        'max_length': max_length,
        'min_score': min_score,
        'max_score': max_score,
        'val_qwk_round': val_eval['qwk_round'],
        'val_qwk': val_eval.get('qwk_opt', np.nan),
        'test_qwk_round': test_eval['qwk_round'],
        'test_qwk': test_eval.get('qwk_opt', np.nan),
        'thresholds': thresholds.tolist(),
    }

    pred_df = te_df.copy()
    pred_df['raw_prediction'] = test_pred_raw
    pred_df['pred_round'] = clip_and_round(test_pred_raw, min_score, max_score)
    pred_df['pred_opt'] = apply_thresholds(test_pred_raw, thresholds, min_score, max_score)
    pred_df['method'] = 'End-to-End Longformer + XHPDF-style Features (ours)'

    hist_df = pd.DataFrame(history)
    hist_df[ESSAY_SET_COL] = prompt_id

    del model
    torch.cuda.empty_cache()

    return result, pred_df, hist_df


In [9]:
# ============================================================
# 8. TRAIN / EVALUATE PER PROMPT
# ============================================================
all_results = []
all_predictions = []
all_histories = []

for prompt_id in sorted(train_df[ESSAY_SET_COL].unique()):
    print('\n' + '=' * 80)
    print(f'PROMPT {prompt_id}: End-to-End Longformer + XHPDF-style Features')
    print('=' * 80)

    tr_idx = train_df[train_df[ESSAY_SET_COL] == prompt_id].index
    va_idx = val_df[val_df[ESSAY_SET_COL] == prompt_id].index
    te_idx = test_df[test_df[ESSAY_SET_COL] == prompt_id].index

    tr_df = train_df.loc[tr_idx].reset_index(drop=True)
    va_df = val_df.loc[va_idx].reset_index(drop=True)
    te_df = test_df.loc[te_idx].reset_index(drop=True)

    tr_feat = train_feat_df.loc[tr_idx].reset_index(drop=True)
    va_feat = val_feat_df.loc[va_idx].reset_index(drop=True)
    te_feat = test_feat_df.loc[te_idx].reset_index(drop=True)

    # Scale features using train only.
    feat_scaler = StandardScaler()
    tr_feat_scaled = feat_scaler.fit_transform(tr_feat.values)
    va_feat_scaled = feat_scaler.transform(va_feat.values)
    te_feat_scaled = feat_scaler.transform(te_feat.values)

    print('Rows train/val/test:', len(tr_df), len(va_df), len(te_df))
    print('Score range:', OFFICIAL_SCORE_RANGES[int(prompt_id)])
    print('Max length:', MAX_LENGTH_BY_PROMPT.get(int(prompt_id), 1024))

    result, pred_df, hist_df = train_one_prompt(
        prompt_id,
        tr_df, va_df, te_df,
        tr_feat_scaled, va_feat_scaled, te_feat_scaled
    )

    all_results.append(result)
    all_predictions.append(pred_df)
    all_histories.append(hist_df)

    print(result)

results_df = pd.DataFrame(all_results)
predictions_df = pd.concat(all_predictions, ignore_index=True)
history_df = pd.concat(all_histories, ignore_index=True)

display(results_df)



PROMPT 1: End-to-End Longformer + XHPDF-style Features
Rows train/val/test: 1248 180 355
Score range: (2, 12)
Max length: 1024


Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prompt 1 epoch 1:   0%|          | 0/78 [00:00<?, ?it/s]

Epoch 1 | train_loss=0.03220 | val_qwk_round=0.72962


...

Prompt 8 epoch 5:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 5 | train_loss=0.00666 | val_qwk_round=0.61012


Prompt 8 epoch 6:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 6 | train_loss=0.00557 | val_qwk_round=0.45782
Early stopping at epoch 6
{'essay_set': 8, 'method': 'End-to-End Longformer + XHPDF-style Features (ours)', 'model_name': 'allenai/longformer-base-4096', 'feature_count': 31, 'max_length': 1536, 'min_score': 0, 'max_score': 60, 'val_qwk_round': np.float64(0.7054562101687021), 'val_qwk': np.float64(0.7246753568289573), 'test_qwk_round': np.float64(0.6765896926005255), 'test_qwk': np.float64(0.6841322140653534), 'thresholds': [0.5010880315062235, 1.5022791986853434, 2.5034417609205097, 3.5052415272565263, 4.506637333903247, 5.511968346568467, 6.508775384020364, 7.516320472593343, 8.518496535605813, 9.520672598618262, 10.52284866163066, 11.517717432976461, 12.5272007876556, 13.529376850668076, 14.531552913680548, 15.533728976692991, 16.535905039705323, 17.53808110271779, 18.526407715754463, 19.542433228742738, 20.544609291755187, 21.546785354767582, 22.548961417780127, 23.551137480792512, 24.553313543805068, 25.555489606817346, 26.53183

,essay_set,method,model_name,feature_count,max_length,min_score,max_score,val_qwk_round,val_qwk,test_qwk_round,test_qwk,thresholds
0,1,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31,1024,2,12,0.850372,0.874724,0.814013,0.824843,"[2.467987024200305, 3.5333878176932414, 4.7468..."
1,2,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31,1024,1,6,0.743995,0.761573,0.722790,0.722816,"[1.5230040000000007, 2.58209, 3.55318599999999..."
2,3,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31,768,0,3,0.673189,0.707818,0.727592,0.730040,"[0.5137602880658434, 1.4977623456790123, 2.368..."
3,4,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31,768,0,3,0.845993,0.878530,0.827653,0.842065,"[0.5757944673068125, 1.3083504801097394, 2.296..."
4,5,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31,768,0,4,0.805389,0.847193,0.800499,0.800569,"[0.5281117708596874, 1.3999563451332504, 2.671..."
5,6,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31,768,0,4,0.809302,0.836577,0.811672,0.808900,"[0.5125, 1.4625, 2.5625, 3.54375]"
6,7,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31,1024,0,30,0.853108,0.859992,0.847795,0.847817,"[0.5015247413380818, 1.5045742240142466, 2.507..."
7,8,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31,1536,0,60,0.705456,0.724675,0.676590,0.684132,"[0.5010880315062235, 1.5022791986853434, 2.503..."


In [10]:
# ============================================================
# 9. AGGREGATE RESULTS AND SAVE
# ============================================================
avg_row = {
    ESSAY_SET_COL: 'Avg',
    'method': 'End-to-End Longformer + XHPDF-style Features (ours)',
    'model_name': MODEL_NAME,
    'feature_count': results_df['feature_count'].mean(),
    'max_length': np.nan,
    'min_score': np.nan,
    'max_score': np.nan,
    'val_qwk_round': results_df['val_qwk_round'].mean(),
    'val_qwk': results_df['val_qwk'].mean(),
    'test_qwk_round': results_df['test_qwk_round'].mean(),
    'test_qwk': results_df['test_qwk'].mean(),
    'thresholds': '',
}

results_with_avg = pd.concat([results_df, pd.DataFrame([avg_row])], ignore_index=True)

results_path = OUTPUT_DIR / 'end2end_longformer_xhpdf_style_features_results.csv'
pred_path = OUTPUT_DIR / 'end2end_longformer_xhpdf_style_features_predictions.csv'
hist_path = OUTPUT_DIR / 'end2end_longformer_xhpdf_style_features_training_history.csv'
feat_path = OUTPUT_DIR / 'end2end_longformer_xhpdf_style_extracted_features.csv'

results_with_avg.to_csv(results_path, index=False)
predictions_df.to_csv(pred_path, index=False)
history_df.to_csv(hist_path, index=False)

features_all = pd.concat([
    pd.concat([train_df[[ESSAY_SET_COL, SCORE_COL]].reset_index(drop=True), train_feat_df.reset_index(drop=True)], axis=1).assign(split='train'),
    pd.concat([val_df[[ESSAY_SET_COL, SCORE_COL]].reset_index(drop=True), val_feat_df.reset_index(drop=True)], axis=1).assign(split='val'),
    pd.concat([test_df[[ESSAY_SET_COL, SCORE_COL]].reset_index(drop=True), test_feat_df.reset_index(drop=True)], axis=1).assign(split='test'),
], ignore_index=True)
features_all.to_csv(feat_path, index=False)

print('Saved:')
print(results_path)
print(pred_path)
print(hist_path)
print(feat_path)

display(results_with_avg)

table_row_round = ['End-to-End Longformer + XHPDF-style Features (ours, round)'] + [
    float(results_df.loc[results_df[ESSAY_SET_COL] == p, 'test_qwk_round'].iloc[0])
    for p in sorted(train_df[ESSAY_SET_COL].unique())
] + [float(results_df['test_qwk_round'].mean())]

table_row_opt = ['End-to-End Longformer + XHPDF-style Features (ours, opt)'] + [
    float(results_df.loc[results_df[ESSAY_SET_COL] == p, 'test_qwk'].iloc[0])
    for p in sorted(train_df[ESSAY_SET_COL].unique())
] + [float(results_df['test_qwk'].mean())]

print('Table row using round:')
print(table_row_round)
print('Table row using opt thresholds:')
print(table_row_opt)


Saved:
/content/benchmark_results/ASAP/end2end_longformer_xhpdf_style_features_results.csv
/content/benchmark_results/ASAP/end2end_longformer_xhpdf_style_features_predictions.csv
/content/benchmark_results/ASAP/end2end_longformer_xhpdf_style_features_training_history.csv
/content/benchmark_results/ASAP/end2end_longformer_xhpdf_style_extracted_features.csv


,essay_set,method,model_name,feature_count,max_length,min_score,max_score,val_qwk_round,val_qwk,test_qwk_round,test_qwk,thresholds
0,1,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31.0,1024.0,2.0,12.0,0.850372,0.874724,0.814013,0.824843,"[2.467987024200305, 3.5333878176932414, 4.7468..."
1,2,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31.0,1024.0,1.0,6.0,0.743995,0.761573,0.722790,0.722816,"[1.5230040000000007, 2.58209, 3.55318599999999..."
2,3,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31.0,768.0,0.0,3.0,0.673189,0.707818,0.727592,0.730040,"[0.5137602880658434, 1.4977623456790123, 2.368..."
3,4,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31.0,768.0,0.0,3.0,0.845993,0.878530,0.827653,0.842065,"[0.5757944673068125, 1.3083504801097394, 2.296..."
4,5,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31.0,768.0,0.0,4.0,0.805389,0.847193,0.800499,0.800569,"[0.5281117708596874, 1.3999563451332504, 2.671..."
5,6,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31.0,768.0,0.0,4.0,0.809302,0.836577,0.811672,0.808900,"[0.5125, 1.4625, 2.5625, 3.54375]"
6,7,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31.0,1024.0,0.0,30.0,0.853108,0.859992,0.847795,0.847817,"[0.5015247413380818, 1.5045742240142466, 2.507..."
7,8,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31.0,1536.0,0.0,60.0,0.705456,0.724675,0.676590,0.684132,"[0.5010880315062235, 1.5022791986853434, 2.503..."
8,Avg,End-to-End Longformer + XHPDF-style Features (...,allenai/longformer-base-4096,31.0,NaN,NaN,NaN,0.785851,0.811385,0.778576,0.782648,


Table row using round:
['End-to-End Longformer + XHPDF-style Features (ours, round)', 0.8140132883354037, 0.7227900891323902, 0.7275921514956112, 0.8276533592989289, 0.8004992381755114, 0.8116719335574385, 0.8477947251365469, 0.6765896926005255, 0.7785755597165446]
Table row using opt thresholds:
['End-to-End Longformer + XHPDF-style Features (ours, opt)', 0.8248426487804257, 0.7228161270616982, 0.7300402424483307, 0.8420646468219197, 0.8005685784697164, 0.808900244863196, 0.8478166907343453, 0.6841322140653534, 0.7826476741556231]


## If Colab runs out of memory

Use these changes in the config cell:

```python
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
MAX_LENGTH_BY_PROMPT[8] = 1024
```

If it is still too slow, run only one prompt first by changing the loop in section 8:

```python
for prompt_id in [1]:
    ...
```

Then restore all prompts when the configuration works.
